# 기초 B4 · 검정이 기대는 가정

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설치는 없고, 구글 계정만 있으면 됩니다.

**읽는 법.** 지금 읽고 있는 이런 글은 설명이라 실행할 것이 없습니다. 코드는 **왼쪽에 `[ ]` 와 ▶ 단추가 붙은 상자**이고, 그 ▶ 또는 `Shift`+`Enter` 로 실행합니다.

**순서.** 번호 차례대로 끝까지 갑니다. **1 준비**부터 시작해 마지막 번호까지 위에서 아래로 내려가면 됩니다. ⚠ 가운데부터 누르면 앞에서 만든 것이 없어 오류가 납니다.

📖 본문 학습 페이지: [기초 B4 · 검정이 기대는 가정](https://grow.minds.kr/textbooks/css-methods/causal/book/b4-가정과-순위-검정.html)

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 2. (책이 원리를 보이려고 돌린 코드) 위양성률을 직접 센다 (책 B4.1 · B4.2)
`위양성`은 세계를 2,000번 지어 p < .05가 나온 비율을 세고, `t검정`은 12장의 t검정, `뒤섞기`는 12장 §12.2의 뒤섞기 검정입니다. `위양성(뽑기, 검정)`은 「세계를 짓는 함수」와 「검정하는 함수」를 인자로 받습니다. 함수도 값처럼 다른 함수에 넘길 수 있어서, 세계와 검정을 바꿔 끼우며 같은 셈을 되풀이할 수 있습니다. 첫 반복 = 치우친 자료(지수분포)에서 집단당 10 · 30 · 100명. 둘째 반복 = 큰 분산이 작은 집단 · 큰 집단에 있는 두 세계에서 t검정 · 웰치 · 뒤섞기(뒤섞기는 오래 걸려 500번). 몇 분 걸립니다.

In [ ]:
from scipy import stats
def 위양성(뽑기, 검정, R=2000, seed=73):
    g = np.random.default_rng(seed)
    hits = 0
    for _ in range(R):
        a, b_ = 뽑기(g)
        hits += 검정(a, b_) < .05
    return round(hits / R, 4)                 # 책보다 한 자리 더 (정확히 반에 걸린 비율이 있다)

t검정 = lambda a, b_: stats.ttest_ind(a, b_).pvalue
웰치 = lambda a, b_: stats.ttest_ind(a, b_, equal_var=False).pvalue

def 뒤섞기(a, b_, B=500):
    g = np.random.default_rng(37)
    obs = abs(a.mean() - b_.mean())
    pool = np.concatenate([a, b_]); n = len(a)
    c = 0
    for _ in range(B):
        p = g.permutation(pool)
        c += abs(p[:n].mean() - p[n:].mean()) >= obs
    return c / B

for n in (10, 30, 100):
    치우침 = lambda g, n=n: (g.exponential(1, n), g.exponential(1, n))
    print(n, 위양성(치우침, t검정), 위양성(치우침, 뒤섞기))
# 10 0.0415 0.0465
# 30 0.0435 0.044
# 100 0.055 0.056

세계 = {"큰 분산이 작은 집단": lambda g: (g.normal(0, 3, 20), g.normal(0, 1, 60)),
        "큰 분산이 큰 집단":   lambda g: (g.normal(0, 1, 20), g.normal(0, 3, 60))}
for 이름, 뽑기 in 세계.items():
    print(이름, 위양성(뽑기, t검정), 위양성(뽑기, 웰치), 위양성(뽑기, 뒤섞기, R=500))
# 큰 분산이 작은 집단 0.222 0.0575 0.22
# 큰 분산이 큰 집단 0.0025 0.0525 0.004

## 3. (책이 원리를 보이려고 돌린 코드) 꼬리가 두꺼우면 비모수 검정의 검정력이 더 높다 (책 B4.4)
극단값이 자주 나오는 분포(자유도 2의 t분포)에서 집단당 40명을 뽑되 한 집단에만 0.8을 더하고, 이 세계를 2,000번 지어 t검정과 맨-휘트니가 각각 효과를 잡은 비율을 셉니다.

In [ ]:
from scipy import stats
g = np.random.default_rng(73)
w1 = w2 = 0
for _ in range(2000):
    u = g.standard_t(2, 40) + 0.8      # 꼬리가 아주 두꺼운 분포
    v = g.standard_t(2, 40)
    w1 += stats.ttest_ind(u, v).pvalue < .05
    w2 += stats.mannwhitneyu(u, v).pvalue < .05
print(round(w1 / 2000, 3), round(w2 / 2000, 3))
# 0.375 0.691

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.